In [0]:
#Ohio Motel Analysis 

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("HotelDataProcessing").getOrCreate()

spark

## 1. Load data

In [0]:
df = spark.table("workspace.default.ohio_house_motel_bookings_3")



In [0]:
%pip install pandas numpy seaborn matplotlib plotly

In [0]:

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt





## 2. Profile the data
Check column types, non-null counts and row count before analyzing anything.

In [0]:
df.printSchema()

In [0]:
df.pandas_api().info()

In [0]:
df.columns

## 3. Clean the data

| Issue | Decision | Why |
|---|---|---|
| Negative ADR or a booking with 0 guests | Remove | Physically impossible, so these are data errors |
| Exact duplicate rows | Remove | `hotel` holds the room number, so two identical rows mean the same room, same dates and same guest details, almost certainly a double-loaded record |
| `company` (mostly empty) and `agent` (an ID with no meaning on its own) | Drop the columns | Not useful for the questions asked here |
| `country` missing | Fill with `"Unknown"` | Keeps the rows; "Unknown" becomes its own category |
| `children` missing (39 rows) | Fill with 0 | Tiny share of rows, and 0 is the most common value |

Section 6 re-runs this whole pipeline and logs how many rows each step removed.

In [0]:
# Impossible values: negative price, or a booking with no guests.
# coalesce() treats a missing children value as 0. Without it, adults + null + babies = null,
# and filter(~bad) silently drops every row where children is null (see section 6.1).
bad = (F.col("adr") < 0) | ((F.col("adults") + F.coalesce(F.col("children"), F.lit(0)) + F.col("babies")) == 0)
print("Impossible rows removed:", df.filter(bad).count())
df = df.filter(~bad)

In [0]:
# Remove exact duplicate rows (same room, dates, guests, price and status)
before = df.count()
df = df.dropDuplicates()
print(f"Duplicate rows removed: {before - df.count():,}")

In [0]:
df.pandas_api().isna().sum()

In [0]:
df = (df.drop("agent", "company")
              .withColumn("children", F.col("children").cast("int"))
              .fillna({"country": "Unknown", "children": 0}))

df.pandas_api().isna().sum()

In [0]:
df.pandas_api().info()

In [0]:
df.select('lead_time').show(5)

## 4. Outliers: do they distort the results?
IQR flags far too many rows on columns that are mostly 0 or small whole numbers (`adults`, `children`, `booking_changes`), so IQR counts alone aren't a reason to delete anything. The real test is whether capping the extremes at the 1st and 99th percentiles changes the averages.

In [0]:
pdf = df.toPandas().select_dtypes("number")
q1, q3 = pdf.quantile(0.25), pdf.quantile(0.75)
iqr = q3 - q1
((pdf < q1 - 1.5 * iqr) | (pdf > q3 + 1.5 * iqr)).sum()

In [0]:
display(df.summary())

In [0]:
for c in ["adr", "lead_time", "stays_in_week_nights"]:
    lo, hi = pdf[c].quantile([0.01, 0.99])
    print(c, "mean:", round(pdf[c].mean(), 2),
          "| capped mean:", round(pdf[c].clip(lo, hi).mean(), 2),
          "| median:", round(pdf[c].median(), 2))

**Outlier conclusion**

| Column | Mean | Capped mean | Change | Decision |
|---|---|---|---|---|
| adr | 102.05 | 101.68 | -0.4% | Keep outliers |
| lead_time | 102.39 | 101.80 | -0.6% | Keep outliers, report the **median** (68 days) |
| stays_in_week_nights | 2.51 | 2.49 | -0.8% | Keep outliers |

Capping moved every average by less than 1%, so outliers are not distorting the results. Lead time has a mean of 102 days and a median of 68: the distribution is skewed (many early bookers), not driven by a few extreme rows, so the median is the honest "typical" value.

## 5. Where guests come from
Guests = adults + children + babies, counted only for bookings that were not canceled.

In [0]:
import plotly.express as px

guests = (df.filter(F.col("is_canceled") == 0)   # only bookings that actually arrived
          .withColumn("guests", F.col("adults") + F.col("children") + F.col("babies"))
          .groupBy("country").agg(F.sum("guests").alias("guests"))
          .orderBy(F.desc("guests"))
          .toPandas())

In [0]:
fig = px.choropleth(guests, locations="country", color="guests",
                    hover_name="country", color_continuous_scale="Blues",
                    title="Number of Guests by Country")
fig.show()

In [0]:
fig2 = px.bar(guests.head(10), x="country", y="guests", text="guests",
              title="Top 10 Countries by Number of Guests")
fig2.show()

## 6. Data checks during analysis
Profiling caught the obvious problems up front. The checks below look for problems that only show up once the analysis is underway: a bug in my own cleaning logic, rules every valid booking must follow, columns that must agree with each other, and gaps in time coverage. Every check prints a count so its impact can be measured.

### 6.1 A bug in my own cleaning filter
The first version of the impossible-values filter was `adults + children + babies == 0`. At that point `children` still had nulls, and in Spark `null + anything = null`. The condition was therefore null for those rows, and `filter(~bad)` drops rows where the condition is null. The 39 rows with missing `children` were being deleted silently instead of filled with 0. Section 3 now uses `coalesce()` to fix it.

In [0]:
RAW_TABLE = "workspace.default.ohio_house_motel_bookings_1"
raw = spark.table(RAW_TABLE)

old_bad = (F.col("adr") < 0) | ((F.col("adults") + F.col("children") + F.col("babies")) == 0)
print("Rows the old filter silently dropped (condition was null):", raw.filter(old_bad.isNull()).count())

### 6.2 Rule checks: what must be true for every booking

In [0]:
nights = F.col("stays_in_weekend_nights") + F.col("stays_in_week_nights")
guests = F.col("adults") + F.col("children") + F.col("babies")

rules = {
    "zero-night stay":                          nights == 0,
    "canceled but status is Check-Out":         (F.col("is_canceled") == 1) & (F.col("reservation_status") == "Check-Out"),
    "not canceled but status Canceled/No-Show": (F.col("is_canceled") == 0) & (F.col("reservation_status") != "Check-Out"),
    "children/babies with no adults":           (F.col("adults") == 0) & (guests > 0),
    "ADR = 0 and not Complementary":            (F.col("adr") == 0) & (F.col("market_segment") != "Complementary"),
}

total = df.count()
counts = df.agg(*[F.sum(cond.cast("int")).alias(name) for name, cond in rules.items()]).first()
rule_checks = pd.DataFrame({"rule": list(rules), "rows": [counts[r] or 0 for r in rules]})
rule_checks["pct_of_rows"] = (100 * rule_checks["rows"] / total).round(2)
rule_checks

**Decisions**
- **Zero-night stays:** remove. No night was sold, so there is no room revenue, and their ADR (often 0) drags price averages down.
- **Status mismatches:** remove. `is_canceled` is the main thing being analyzed, so rows where it contradicts the reservation status can't be trusted.
- **Children with no adults, and ADR = 0 outside Complementary:** keep. Both can happen in real life (a family booking under a child's name, staff or promotional stays), and they are a small share of rows.

In [0]:
status_mismatch = rules["canceled but status is Check-Out"] | rules["not canceled but status Canceled/No-Show"]

before = df.count()
df = df.filter((nights > 0) & ~status_mismatch)
print(f"Removed {before - df.count():,} rows ({100 * (before - df.count()) / before:.2f}%) | rows left: {df.count():,}")

### 6.3 Time coverage: are all years complete?

In [0]:
coverage = (df.groupBy("arrival_date_year")
              .agg(F.countDistinct("arrival_date_month").alias("months_of_data"),
                   F.count("*").alias("bookings"))
              .orderBy("arrival_date_year"))
display(coverage)

**Finding:** 2015 only has data from July and 2017 stops in August, so only 2016 is a full year. Comparing yearly totals would make 2016 look like a boom year and 2017 look like a decline, when those years just have fewer months of data.

**Fix:** in the seasonality analysis (8.2), monthly figures are divided by the number of years of data for that month, so July and August (3 years of data) are compared fairly against other months (2 years).

### 6.4 Cleaning log: how many rows each step removed
The function below re-runs every cleaning step from the raw table, so the whole pipeline is in one place and each step's impact is recorded.

In [0]:
def clean_pipeline(raw, dedup=True, fill_nulls=True, track=True):
    """Run every cleaning step from the raw table. Returns (clean_df, [(step, rows_after), ...])."""
    steps = []
    log = lambda name, d: steps.append((name, d.count())) if track else None

    d = raw
    log("raw data", d)

    no_guests = (F.col("adults") + F.coalesce(F.col("children"), F.lit(0)) + F.col("babies")) == 0
    d = d.filter(~((F.col("adr") < 0) | no_guests))
    log("remove negative ADR / zero guests", d)

    if dedup:
        d = d.dropDuplicates()
        log("remove duplicate rows", d)

    d = d.drop("agent", "company").withColumn("children", F.col("children").cast("int"))
    if fill_nulls:
        d = d.fillna({"country": "Unknown", "children": 0})
    else:
        d = d.dropna(subset=["country", "children"])
        log("drop rows with null country/children", d)

    n = F.col("stays_in_weekend_nights") + F.col("stays_in_week_nights")
    status_ok = (((F.col("is_canceled") == 1) & (F.col("reservation_status") != "Check-Out")) |
                 ((F.col("is_canceled") == 0) & (F.col("reservation_status") == "Check-Out")))
    d = d.filter((n > 0) & status_ok)
    log("remove zero-night stays / status mismatches", d)
    return d, steps


_, steps = clean_pipeline(raw)
cleaning_log = pd.DataFrame(steps, columns=["step", "rows_after"])
cleaning_log["rows_removed"] = (-cleaning_log["rows_after"].diff()).fillna(0).astype(int)
cleaning_log["pct_of_raw"] = (100 * cleaning_log["rows_removed"] / cleaning_log.loc[0, "rows_after"]).round(2)
print("Pipeline result matches the notebook's df:", cleaning_log["rows_after"].iloc[-1] == df.count())
cleaning_log

### 6.5 Sensitivity check: do the cleaning choices change the answer?
The same key metrics are calculated under different cleaning choices. If they barely move, the conclusions don't depend on those choices. If they move a lot, that needs to be reported.

In [0]:
def key_metrics(d):
    r = d.agg(F.count("*").alias("rows"),
              F.avg("is_canceled").alias("cancel_rate"),
              F.avg("adr").alias("avg_adr")).first()
    return r["rows"], round(100 * r["cancel_rate"], 2), round(r["avg_adr"], 2)

variants = {
    "final (dedup + fill nulls)":     clean_pipeline(raw, track=False)[0],
    "keep duplicates":                clean_pipeline(raw, dedup=False, track=False)[0],
    "drop null rows instead of fill": clean_pipeline(raw, fill_nulls=False, track=False)[0],
}
sensitivity = pd.DataFrame([(name, *key_metrics(d)) for name, d in variants.items()],
                           columns=["cleaning choice", "rows", "cancel_rate_pct", "avg_adr"])
base = sensitivity.iloc[0]
sensitivity["cancel_rate_change_pts"] = (sensitivity["cancel_rate_pct"] - base["cancel_rate_pct"]).round(2)
sensitivity["avg_adr_change_pct"] = (100 * (sensitivity["avg_adr"] / base["avg_adr"] - 1)).round(2)
sensitivity

## 7. Feature engineering
New columns used in the revenue analysis:
- `month_num`, `arrival_date`: real dates, so months sort in calendar order
- `total_nights`, `total_guests`
- `revenue` = ADR × nights, the estimated room revenue of the booking (meals and extras are not in the data)
- `realized_revenue`: revenue from bookings that were not canceled
- `lost_revenue`: revenue from bookings that were canceled
- `room_changed`: the guest got a different room type than they reserved
- `lead_time_bucket`: how far ahead the booking was made

Note: in this version of the dataset the `hotel` column holds room numbers ("Room 328"), not hotel type, so it is treated as a room ID.

In [0]:
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

MONTHS = ["January", "February", "March", "April", "May", "June",
          "July", "August", "September", "October", "November", "December"]
LEAD_ORDER = ["0-7 days", "8-30 days", "31-90 days", "91-180 days", "181-365 days", "365+ days"]
month_map = F.create_map(*[x for i, m in enumerate(MONTHS, 1) for x in (F.lit(m), F.lit(i))])

df = (df
      .withColumn("month_num", month_map[F.col("arrival_date_month")])
      .withColumn("arrival_date", F.expr("make_date(arrival_date_year, month_num, arrival_date_day_of_month)"))
      .withColumn("total_nights", F.col("stays_in_weekend_nights") + F.col("stays_in_week_nights"))
      .withColumn("total_guests", F.col("adults") + F.col("children") + F.col("babies"))
      .withColumn("revenue", F.round(F.col("adr") * F.col("total_nights"), 2))
      .withColumn("realized_revenue", F.when(F.col("is_canceled") == 0, F.col("revenue")).otherwise(F.lit(0.0)))
      .withColumn("lost_revenue", F.when(F.col("is_canceled") == 1, F.col("revenue")).otherwise(F.lit(0.0)))
      .withColumn("room_changed", (F.col("reserved_room_type") != F.col("assigned_room_type")).cast("int"))
      .withColumn("lead_time_bucket",
                  F.when(F.col("lead_time") <= 7, LEAD_ORDER[0])
                   .when(F.col("lead_time") <= 30, LEAD_ORDER[1])
                   .when(F.col("lead_time") <= 90, LEAD_ORDER[2])
                   .when(F.col("lead_time") <= 180, LEAD_ORDER[3])
                   .when(F.col("lead_time") <= 365, LEAD_ORDER[4])
                   .otherwise(LEAD_ORDER[5])))

print(df.agg(F.min("arrival_date").alias("first"), F.max("arrival_date").alias("last")).first())
display(df.select("arrival_date", "total_nights", "adr", "revenue", "realized_revenue",
                  "lost_revenue", "room_changed", "lead_time_bucket").limit(10))

## 8. Revenue analysis
`seg_summary()` gives the same scorecard for any column, so each question below is one call:
bookings, cancellation rate, average ADR, realized revenue, lost revenue, and **revenue per booking made** (realized revenue ÷ all bookings, including canceled ones). That last metric shows what a booking from a segment is actually worth once its cancellations are taken into account.

In [0]:
def seg_summary(data, group_col, min_bookings=0):
    return (data.groupBy(group_col)
                .agg(F.count("*").alias("bookings"),
                     F.round(100 * F.avg("is_canceled"), 1).alias("cancel_rate_pct"),
                     F.round(F.avg("adr"), 2).alias("avg_adr"),
                     F.round(F.sum("realized_revenue"), 0).alias("realized_revenue"),
                     F.round(F.sum("lost_revenue"), 0).alias("lost_revenue"))
                .withColumn("revenue_per_booking", F.round(F.col("realized_revenue") / F.col("bookings"), 2))
                .filter(F.col("bookings") >= min_bookings))

### 8.1 Cancellations and lost revenue
Which bookings cancel, and how much revenue do cancellations cost?

In [0]:
kpi = df.agg(F.count("*").alias("bookings"),
             F.avg("is_canceled").alias("cancel_rate"),
             F.sum("realized_revenue").alias("realized"),
             F.sum("lost_revenue").alias("lost")).first()

print(f"Bookings:                    {kpi['bookings']:,}")
print(f"Cancellation rate:           {100 * kpi['cancel_rate']:.1f}%")
print(f"Realized room revenue:       ${kpi['realized']:,.0f}")
print(f"Revenue lost to cancels:     ${kpi['lost']:,.0f} "
      f"({100 * kpi['lost'] / (kpi['realized'] + kpi['lost']):.1f}% of all booked revenue)")

In [0]:
segment_df = seg_summary(df, "market_segment").orderBy(F.desc("lost_revenue")).toPandas()
display(segment_df)

fig = px.bar(segment_df, x="market_segment", y="lost_revenue", text_auto=".3s",
             hover_data=["bookings", "cancel_rate_pct"],
             title="Revenue lost to cancellations by market segment")
fig.update_layout(xaxis_title=None, yaxis_title="Lost revenue ($)")
fig.show()

In [0]:
lead_df = seg_summary(df, "lead_time_bucket").toPandas()
lead_df["lead_time_bucket"] = pd.Categorical(lead_df["lead_time_bucket"], LEAD_ORDER, ordered=True)
lead_df = lead_df.sort_values("lead_time_bucket")
display(lead_df)

fig = px.bar(lead_df, x="lead_time_bucket", y="cancel_rate_pct", text="cancel_rate_pct",
             title="Cancellation rate by how far ahead the booking was made")
fig.update_layout(xaxis_title="Lead time", yaxis_title="Cancellation rate (%)")
fig.show()

In [0]:
# Other cancellation drivers
deposit_df = seg_summary(df, "deposit_type").orderBy(F.desc("cancel_rate_pct")).toPandas()
display(deposit_df)

display(seg_summary(df, "customer_type").orderBy(F.desc("cancel_rate_pct")))

display(seg_summary(df.withColumn("has_cancelled_before", F.col("previous_cancellations") > 0),
                    "has_cancelled_before"))

**Watch out with deposit type:** if "Non Refund" bookings show a very high cancellation rate, that "lost" revenue was mostly **kept** by the hotel, since the deposit isn't returned. The lost-revenue figure is therefore an upper bound. The real loss is concentrated in "No Deposit" bookings.

### 8.2 Seasonality: when is demand high, and is price keeping up?
Monthly figures are divided by the number of years of data for each month (the partial-year issue from 6.3).

In [0]:
monthly_df = (df.groupBy("month_num", "arrival_date_month")
                .agg(F.countDistinct("arrival_date_year").alias("years_of_data"),
                     F.sum(1 - F.col("is_canceled")).alias("arrivals"),
                     F.round(F.avg("adr"), 2).alias("avg_adr"),
                     F.round(100 * F.avg("is_canceled"), 1).alias("cancel_rate_pct"),
                     F.sum("realized_revenue").alias("realized_revenue"))
                .withColumn("arrivals_per_year", F.round(F.col("arrivals") / F.col("years_of_data"), 0))
                .withColumn("revenue_per_year", F.round(F.col("realized_revenue") / F.col("years_of_data"), 0))
                .orderBy("month_num")
                .toPandas())
display(monthly_df)

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=monthly_df["arrival_date_month"], y=monthly_df["arrivals_per_year"],
                     name="Arrivals per year"), secondary_y=False)
fig.add_trace(go.Scatter(x=monthly_df["arrival_date_month"], y=monthly_df["avg_adr"],
                         name="Average ADR ($)", mode="lines+markers"), secondary_y=True)
fig.update_layout(title="Demand vs. price by month")
fig.update_yaxes(title_text="Arrivals per year", secondary_y=False)
fig.update_yaxes(title_text="Average ADR ($)", secondary_y=True)
fig.show()

### 8.3 Channel and segment profitability
A segment with a high price but a high cancellation rate can be worth less than it looks. Bubbles in the top-left of the chart are the best: high price and low cancellations.

In [0]:
fig = px.scatter(segment_df, x="cancel_rate_pct", y="avg_adr", size="bookings",
                 color="market_segment", text="market_segment",
                 title="Market segments: price vs. cancellation risk (bubble size = bookings)")
fig.update_traces(textposition="top center")
fig.update_layout(xaxis_title="Cancellation rate (%)", yaxis_title="Average ADR ($)", showlegend=False)
fig.show()

display(segment_df.sort_values("revenue_per_booking", ascending=False))
display(seg_summary(df, "distribution_channel").orderBy(F.desc("revenue_per_booking")))

### 8.4 Upselling: room types, special requests, meals and parking

In [0]:
room_df = seg_summary(df, "room_changed").toPandas()
print(f"Bookings that got a different room type than reserved: {100 * df.agg(F.avg('room_changed')).first()[0]:.1f}%")
display(room_df)

requests_df = seg_summary(df, "total_of_special_requests").orderBy("total_of_special_requests").toPandas()
fig = px.bar(requests_df, x="total_of_special_requests", y="cancel_rate_pct", text="cancel_rate_pct",
             title="Cancellation rate by number of special requests")
fig.update_layout(xaxis_title="Special requests", yaxis_title="Cancellation rate (%)")
fig.show()

display(seg_summary(df, "meal").orderBy(F.desc("bookings")))
display(seg_summary(df, "required_car_parking_spaces").orderBy("required_car_parking_spaces"))

**Caution:** room types are usually assigned at check-in, and canceled bookings never check in. Room changes will therefore look strongly linked to low cancellation even though the change doesn't cause it. Use this table to see how often upgrades happen (an upsell opportunity), not as a cancellation driver.

### 8.5 Length of stay by customer type (stays that happened)

In [0]:
stay_df = (df.filter(F.col("is_canceled") == 0)
             .groupBy("customer_type")
             .agg(F.count("*").alias("stays"),
                  F.round(F.avg("total_nights"), 2).alias("avg_nights"),
                  F.round(100 * F.avg(F.col("stays_in_weekend_nights") / F.col("total_nights")), 1).alias("weekend_night_share_pct"),
                  F.round(F.avg("adr"), 2).alias("avg_adr"),
                  F.round(F.avg("revenue"), 2).alias("revenue_per_stay"))
             .orderBy(F.desc("revenue_per_stay"))
             .toPandas())
display(stay_df)

fig = px.bar(stay_df, x="customer_type", y="revenue_per_stay", text_auto=".0f",
             hover_data=["avg_nights", "avg_adr", "stays"],
             title="Average revenue per stay by customer type")
fig.update_layout(xaxis_title=None, yaxis_title="Revenue per stay ($)")
fig.show()

### 8.6 Repeat guests

In [0]:
repeat_df = seg_summary(df, "is_repeated_guest").orderBy("is_repeated_guest").toPandas()
repeat_df["share_of_bookings_pct"] = (100 * repeat_df["bookings"] / repeat_df["bookings"].sum()).round(1)
display(repeat_df)

### 8.7 Countries: high value and reliable
The guest map in section 5 shows volume. This shows which countries bring revenue **and** follow through on their bookings. Only countries with at least `MIN_COUNTRY_BOOKINGS` bookings are included, so small countries don't produce misleading rates.

In [0]:
MIN_COUNTRY_BOOKINGS = 1000

country_df = (seg_summary(df, "country", min_bookings=MIN_COUNTRY_BOOKINGS)
                .orderBy(F.desc("realized_revenue")).limit(15).toPandas())
display(country_df)

fig = px.scatter(country_df, x="realized_revenue", y="cancel_rate_pct", size="bookings", text="country",
                 title="Top 15 countries: revenue vs. cancellation rate")
fig.update_traces(textposition="top center")
fig.update_layout(xaxis_title="Realized revenue ($)", yaxis_title="Cancellation rate (%)")
fig.show()

### 8.8 Lead time vs. price: do early bookers pay more or less?

In [0]:
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=lead_df["lead_time_bucket"].astype(str), y=lead_df["avg_adr"], name="Average ADR ($)"),
              secondary_y=False)
fig.add_trace(go.Scatter(x=lead_df["lead_time_bucket"].astype(str), y=lead_df["cancel_rate_pct"],
                         name="Cancellation rate (%)", mode="lines+markers"), secondary_y=True)
fig.update_layout(title="Price and cancellation rate by lead time")
fig.update_yaxes(title_text="Average ADR ($)", secondary_y=False)
fig.update_yaxes(title_text="Cancellation rate (%)", secondary_y=True)
fig.show()

## 9. Key insights
Generated from the tables above, so the numbers always match the data.

In [0]:
booked_total = kpi["realized"] + kpi["lost"]
seg_top = segment_df.iloc[0]
lead_hi, lead_lo = lead_df.loc[lead_df["cancel_rate_pct"].idxmax()], lead_df.loc[lead_df["cancel_rate_pct"].idxmin()]
adr_hi, adr_lo = monthly_df.loc[monthly_df["avg_adr"].idxmax()], monthly_df.loc[monthly_df["avg_adr"].idxmin()]
busy = monthly_df.loc[monthly_df["arrivals_per_year"].idxmax()]
busy_raw = monthly_df.loc[monthly_df["arrivals"].idxmax()]
best_seg = segment_df[segment_df["bookings"] >= 0.01 * kpi["bookings"]].sort_values("revenue_per_booking").iloc[-1]
new, rep = repeat_df.set_index("is_repeated_guest").loc[0], repeat_df.set_index("is_repeated_guest").loc[1]
top_country = country_df.iloc[0]
no_dep = deposit_df.set_index("deposit_type")

insights = [
    f"{100 * kpi['cancel_rate']:.1f}% of bookings are canceled, costing ${kpi['lost']:,.0f} "
    f"({100 * kpi['lost'] / booked_total:.1f}% of all booked room revenue).",
    f"Non Refund bookings make up ${no_dep.loc['Non Refund', 'lost_revenue']:,.0f} of that, but the hotel keeps those deposits, "
    f"so the real loss is about ${kpi['lost'] - no_dep.loc['Non Refund', 'lost_revenue']:,.0f}.",
    f"{seg_top['market_segment']} accounts for {100 * seg_top['lost_revenue'] / kpi['lost']:.0f}% of lost revenue "
    f"(cancellation rate {seg_top['cancel_rate_pct']}%).",
    f"Cancellation rate is lowest for bookings made {lead_lo['lead_time_bucket']} ahead ({lead_lo['cancel_rate_pct']}%) "
    f"and highest for {lead_hi['lead_time_bucket']} ahead ({lead_hi['cancel_rate_pct']}%).",
    "Cancellation rate by deposit type: " + ", ".join(f"{k} {v}%" for k, v in no_dep["cancel_rate_pct"].items()) + ".",
    f"Prices peak in {adr_hi['arrival_date_month']} (avg ADR ${adr_hi['avg_adr']:.0f}) and bottom out in "
    f"{adr_lo['arrival_date_month']} (${adr_lo['avg_adr']:.0f}). After adjusting for partial years, the busiest "
    f"month is {busy['arrival_date_month']}" + (f" (raw totals would wrongly say {busy_raw['arrival_date_month']})."
                                                 if busy_raw['arrival_date_month'] != busy['arrival_date_month'] else "."),
    f"{best_seg['market_segment']} brings the most revenue per booking made (${best_seg['revenue_per_booking']:.0f}).",
    f"Repeat guests are {repeat_df.loc[repeat_df['is_repeated_guest'] == 1, 'share_of_bookings_pct'].iloc[0]}% of bookings "
    f"and cancel {rep['cancel_rate_pct']}% of the time vs {new['cancel_rate_pct']}% for new guests.",
    f"{top_country['country']} is the top revenue country (${top_country['realized_revenue']:,.0f}) "
    f"with a {top_country['cancel_rate_pct']}% cancellation rate.",
]
for i, s in enumerate(insights, 1):
    print(f"{i}. {s}")